# 06 — Model Explainability (SHAP)

**Goal:** Use SHAP to explain model predictions — both globally (which features matter most) and individually (why a specific customer was flagged).

> **Important:** SHAP describes how the model makes its decisions. It does NOT prove that a feature *causes* churn.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import (
    load_raw_data, clean_data, separate_features_target, split_data,
    identify_column_types, build_preprocessor, TARGET_COL
)
from src.feature_engineering import engineer_features
from src.train import load_model
from src.explain import (
    get_shap_explainer, plot_shap_summary, plot_shap_bar,
    explain_individual, get_top_features
)

## 6.1 Load Model and Prepare Data

In [ ]:
# Load saved model
pipeline = load_model()

# Prepare data (same pipeline as training)
df_raw = load_raw_data()
df = clean_data(df_raw)
df = engineer_features(df)
X, y, ids = separate_features_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Using test set ({len(X_test)} rows) for SHAP explanations")

## 6.2 Generate SHAP Values

In [ ]:
# Use a sample for SHAP (full test set or a subset)
sample_size = min(500, len(X_test))
X_sample = X_test.iloc[:sample_size]

explainer, shap_values, X_transformed_df = get_shap_explainer(pipeline, X_sample)
print(f"SHAP values shape: {shap_values.shape}")
print(f"Transformed features: {X_transformed_df.shape[1]}")

## 6.3 Global Feature Importance

In [ ]:
# SHAP Summary (Beeswarm) Plot
fig = plot_shap_summary(shap_values, X_transformed_df)
plt.show()

In [ ]:
# SHAP Bar Plot (Mean |SHAP|)
fig = plot_shap_bar(shap_values, X_transformed_df)
plt.show()

In [ ]:
# Top features table
top_features = get_top_features(shap_values, X_transformed_df.columns.tolist(), top_n=15)
print("\nTop 15 Features by Mean |SHAP Value|:")
print(top_features.to_string(index=False))

## 6.4 Individual Customer Explanations

Explain predictions for specific customers to understand why the model assigned a particular churn probability.

In [ ]:
# Get probabilities for the sample
y_sample_prob = pipeline.predict_proba(X_sample)[:, 1]

# Find a high-risk and low-risk customer
high_risk_idx = np.argmax(y_sample_prob)
low_risk_idx = np.argmin(y_sample_prob)

print(f"High-risk customer (index {high_risk_idx}): probability = {y_sample_prob[high_risk_idx]:.4f}")
print(f"Low-risk customer (index {low_risk_idx}): probability = {y_sample_prob[low_risk_idx]:.4f}")

In [ ]:
# Explain high-risk customer
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1] if len(expected_value) > 1 else expected_value[0]

print("\n--- High-Risk Customer Explanation ---")
fig = explain_individual(high_risk_idx, shap_values, X_transformed_df, expected_value)
plt.show()

In [ ]:
# Explain low-risk customer
print("\n--- Low-Risk Customer Explanation ---")
fig = explain_individual(low_risk_idx, shap_values, X_transformed_df, expected_value)
plt.show()

## 6.5 Business Interpretation

### Global Insights (from SHAP)

The model relies most heavily on:
1. **Contract type** — Month-to-month contracts push predictions toward churn.
2. **Tenure** — Shorter tenure increases predicted churn probability.
3. **Monthly charges** — Higher charges correlate with higher predicted churn.
4. **Internet service type** — Fiber optic customers tend to have higher predicted churn.
5. **Tech support / Online security** — Absence of these services increases predicted churn.

### Reminders

- These are **model-derived patterns**, not proven causes.
- A feature being important to the model does not mean changing it will reduce churn.
- Interventions should be tested through controlled experiments before claiming effectiveness.